# DRCFNet Model - CMU-MOSI Dataset

Dynamic Reconstructive Context Fusion Network for Sentiment Analysis

In [1]:
# !git clone https://github.com/ranbeer06052009/MulG-Research

Cloning into 'MulG-Research'...
remote: Enumerating objects: 242, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 242 (delta 48), reused 104 (delta 31), pack-reused 118 (from 1)
Receiving objects: 100% (242/242), 341.60 MiB | 22.90 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Updating files: 100% (126/126), done.


In [2]:
import gdown

file_id = "1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU"
destination = "mosi_raw.pkl"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU
From (redirected): https://drive.google.com/uc?id=1szKIqO0t3Be_W91xvf6aYmsVVUa7wDHU&confirm=t&uuid=07098ca4-abea-489b-a254-b193e6aff981
To: /content/mosi_raw.pkl
100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 357M/357M [00:01<00:00, 218MB/s]


'mosi_raw.pkl'

In [3]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.append('/content/MulG-Research/src')

from loader import get_dataloader
from models.drcfnet import DRCFNet
from objectives import NeuroSymbolicLoss
from training.train import train, test
from evaluation.performance import eval_affect

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: cuda


In [4]:
# Load MOSI Data
train_data, valid_data, test_data = get_dataloader(
    '/content/mosi_raw.pkl',
    max_pad=True,
    max_seq_len=50
)

In [5]:
# Initialize Model and Loss
model = DRCFNet(dim_v=35, dim_a=74, dim_t=300, d=128, n_heads=4, dropout=0.2).to(device)
criterion = NeuroSymbolicLoss(lambda_logic=0.1, lambda_task=1.0).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

In [6]:
# Initialize Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)

# Train the model
model, history = train(
    model=model,
    train_loader=train_data,
    valid_loader=valid_data,
    criterion=criterion,
    optimizer=optimizer,
    epochs=50,
    device=device,
    scheduler=scheduler,
    clip_grad=1.0
)

Epoch 1 | Train Loss: 1.3386 (Task:1.3038 Orth:0.2849 Contr:3.1930)
Epoch 1 | Valid Loss: 1.4147 (Task:1.3812 Orth:0.4308 Contr:2.9110)


Epoch 2 | Train Loss: 1.2950 (Task:1.2636 Orth:0.2422 Contr:2.9019)
Epoch 2 | Valid Loss: 1.2871 (Task:1.2494 Orth:0.4493 Contr:3.3249)


Epoch 3 | Train Loss: 1.0976 (Task:1.0628 Orth:0.2932 Contr:3.1853)
Epoch 3 | Valid Loss: 1.1202 (Task:1.0833 Orth:0.4750 Contr:3.2180)


Epoch 4 | Train Loss: 0.9607 (Task:0.9257 Orth:0.3205 Contr:3.1751)
Epoch 4 | Valid Loss: 1.0913 (Task:1.0545 Orth:0.4759 Contr:3.2054)


Epoch 5 | Train Loss: 0.8892 (Task:0.8548 Orth:0.3221 Contr:3.1183)
Epoch 5 | Valid Loss: 1.0235 (Task:0.9872 Orth:0.4772 Contr:3.1495)


...

Epoch 96 | Train Loss: 0.3262 (Task:0.2943 Orth:0.2586 Contr:2.9306)
Epoch 96 | Valid Loss: 0.9318 (Task:0.8973 Orth:0.4185 Contr:3.0259)


Epoch 97 | Train Loss: 0.3115 (Task:0.2794 Orth:0.2607 Contr:2.9494)
Epoch 97 | Valid Loss: 0.9318 (Task:0.8974 Orth:0.4185 Contr:3.0259)


Epoch 98 | Train Loss: 0.3122 (Task:0.2803 Orth:0.2593 Contr:2.9344)
Epoch 98 | Valid Loss: 0.9318 (Task:0.8974 Orth:0.4185 Contr:3.0259)


Epoch 99 | Train Loss: 0.3203 (Task:0.2884 Orth:0.2592 Contr:2.9312)
Epoch 99 | Valid Loss: 0.9318 (Task:0.8974 Orth:0.4185 Contr:3.0259)


Epoch 100 | Train Loss: 0.3170 (Task:0.2853 Orth:0.2585 Contr:2.9125)
Epoch 100 | Valid Loss: 0.9318 (Task:0.8974 Orth:0.4185 Contr:3.0259)


In [7]:
import numpy as np
from sklearn.metrics import f1_score
# Test the model
test_loss, preds, labels = test(
    model=model,
    dataloader=test_data,
    criterion=criterion,
    device=device,
    return_preds=True
)

preds_np = preds.view(-1).cpu().numpy()
labels_np = labels.view(-1).cpu().numpy()

# ===== MAE =====
mae = np.mean(np.abs(preds_np - labels_np))

# ===== Corr =====
corr = np.corrcoef(preds_np, labels_np)[0][1]
corr = 0 if np.isnan(corr) else corr

# ===== Binary =====
pred_bin = (preds_np > 0).astype(int)
label_bin = (labels_np > 0).astype(int)

mask = labels_np != 0
acc2 = (pred_bin[mask] == label_bin[mask]).mean()

f1 = f1_score(label_bin[mask], pred_bin[mask], average='weighted')

print("\nFinal Evaluation:")
print(f"MAE: {mae:.4f}")
print(f"Corr: {corr:.4f}")
print(f"Acc-2: {acc2*100:.2f}%")
print(f"F1: {f1*100:.2f}%")


Final Evaluation:
MAE: 1.0642
Corr: 0.5729
Acc-2: 71.34%
F1: 71.49%


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["valid_loss"], label="Valid Loss")
plt.title("Total Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_task"], label="Train MAE")
plt.plot(history["valid_task"], label="Valid MAE")
plt.title("Task MAE (Regression)")
plt.legend()
plt.show()

In [ ]:
# Save the model
torch.save(model.state_dict(), "drcfnet_mosi_best.pt")
print("Model saved as drcfnet_mosi_best.pt")